# Foundations of R Software — Week 10
## Strings (Substitution & Searching) and Data Frames

**Source:** NPTEL, Prof. Shalabh (IIT Kanpur), Lectures 40–44

| Lecture | Topic | Functions |
|---|---|---|
| 40 | Substitution and searching in strings | `sub`, `gsub`, `grep`, `grepl` |
| 41 | Data frames — introduction | `data.frame`, `$`, `is.numeric`, `is.factor`, `colnames`, `rownames`, `summary` |
| 42 | Creation and operations | the `painters` data set from **MASS** |
| 43 | More operations | `summary`, `attach`/`detach`, `subset`, `split` |
| 44 | Combining and merging | `cbind`, `merge`, `rbind` |

> **Colab tip:** use the **R** runtime (*Runtime → Change runtime type → R*), or start from  
> `https://colab.research.google.com/notebook#create=true&language=r`.  
> The **MASS** package ships with R, so nothing needs installing.

In [ ]:
R.version.string

[1] "R version 4.6.1 (2026-06-24)"

---
# Lecture 40 — Substitution and Searching in Strings

R has many functions for regular-expression based matching:

* **`sub`, `gsub`** → *replace* text
* **`grep`, `grepl`** → *search* for matches

## 1. `sub()` and `gsub()` — replacement

```r
sub(old, new, string)     # replaces the FIRST instance
gsub(old, new, string)    # replaces ALL instances (global replace)
```

### Simple replacement

In [ ]:
y = "Number of participants: 25"
sub("25", "30", y)

[1] "Number of participants: 30"

### `sub()` changes only the first match

In [ ]:
y = "Mr. Singh is the smart one. Mr. Singh is funny, too."
y
sub("Mr. Singh", "Professor Jha", y)

[1] "Mr. Singh is the smart one. Mr. Singh is funny, too."

[1] "Professor Jha is the smart one. Mr. Singh is funny, too."

### `gsub()` changes every match

In [ ]:
gsub("Mr. Singh", "Professor Jha", y)

# Recall:
sub("Mr. Singh", "Professor Jha", y)

[1] "Professor Jha is the smart one. Professor Jha is funny, too."

[1] "Professor Jha is the smart one. Mr. Singh is funny, too."

> 💡 **Note:** the first argument is a *regular expression*, so a `.` matches **any** character. To treat the pattern as plain text, add `fixed = TRUE`, e.g. `gsub(".", "-", "a.b.c", fixed = TRUE)`.

In [ ]:
gsub(".", "-", "a.b.c")                # regex: every character matches
gsub(".", "-", "a.b.c", fixed = TRUE)  # literal dot only

[1] "-----"

[1] "a-b-c"

## 2. `grep()` — searching for matches

*grep* = **G**lobally search a **R**egular **E**xpression and **P**rint it.

```r
grep(pattern, x, ignore.case = FALSE, value = FALSE)
```

| Argument | Meaning |
|---|---|
| `ignore.case` | `FALSE` (default): case-sensitive. `TRUE`: ignore case |
| `value` | `FALSE` (default): return the **indices** of matches. `TRUE`: return the **matching elements** |

### `value = TRUE` → the matching elements

In [ ]:
str = c("R Course", "exercises", "include examples of R language")
grep("ex", str, value = TRUE)     # the slide writes value=T; TRUE is safer

[1] "exercises"                      "include examples of R language"

### `value = FALSE` (default) → indices

In [ ]:
grep("ex", str, value = FALSE)

[1] 2 3

### `ignore.case`

In [ ]:
str = c("R Course", "exercises", "include examples of r language", "in R software.")

grep("R", str, ignore.case = FALSE, value = TRUE)
grep("R", str, ignore.case = TRUE,  value = TRUE)

[1] "R Course"       "in R software."

[1] "R Course"                       "exercises"                     
[3] "include examples of r language" "in R software."

In [ ]:
grep("R", str, ignore.case = TRUE,  value = FALSE)
grep("R", str, ignore.case = FALSE, value = FALSE)

[1] 1 2 3 4

[1] 1 4

### Searching across several strings
`"our"` appears in *course* (element 1) but not in the second string.

In [ ]:
x = "R course 24.07.2021"
y = "Number of participants: 25"
c(x, y)                 # combine the two strings
grep("our", c(x, y))

[1] "R course 24.07.2021"        "Number of participants: 25"

[1] 1

`"Num"` appears in *Number* (element 2) but not in the first string.

In [ ]:
x = "R course 24.07.2022"
y = "Number of participants: 50"
c(x, y)
grep("Num", c(x, y))

[1] "R course 24.07.2022"        "Number of participants: 50"

[1] 2

## 3. `grepl()` — TRUE/FALSE matching

```r
grepl(pattern, x)
```
Returns a **logical vector** (same length as `x`): `TRUE` where the pattern matches, `FALSE` otherwise.

In [ ]:
str = c("R Course", "exercises", "include examples of R language")
str
grepl("R", str)
grepl("ex", str)

[1] "R Course"                       "exercises"                     
[3] "include examples of R language"

[1]  TRUE FALSE  TRUE

[1] FALSE  TRUE  TRUE

> ⚠️ `grepl()` has **no `value` argument** — the slide text mentions one, but the console output on the slides uses plain `grepl("ex", str)`. Trying `value = TRUE` raises an error:

In [ ]:
tryCatch(
  grepl("ex", str, value = TRUE),
  error = function(e) cat("Error:", conditionMessage(e), "\n")
)

Error: unused argument (value = TRUE) 


**`grep` vs `grepl` at a glance**

| | Returns | Example result |
|---|---|---|
| `grep("ex", str)` | indices | `2 3` |
| `grep("ex", str, value=TRUE)` | matching strings | `"exercises" "include examples..."` |
| `grepl("ex", str)` | TRUE/FALSE per element | `FALSE TRUE TRUE` |

---
# Lecture 41 — Data Frames: Introduction

`c`, `cbind`, `vector` and `matrix` combine data. Another option is the **data frame**:

* combines variables of **equal length**, each row = observations on the same unit (like a matrix / `cbind`)
* can mix **numeric, character and factor** columns — `cbind()` and `matrix()` cannot
* works like a spreadsheet: **columns = variables**, **rows = observations**
* changes can be made without affecting the original data
* usually created from other sources (spreadsheets, SPSS files, Excel files…)

### Why not `cbind` / `matrix` for mixed data?
A matrix has a single type, so mixed columns are coerced to text; a data frame keeps each column's type.

In [ ]:
name  = c("Asha", "Ravi", "Meena")
marks = c(78, 91, 85)

m  = cbind(name, marks)        # everything becomes character
m
class(m[, "marks"])

df = data.frame(name, marks)   # each column keeps its own type
df
class(df$marks)

name,marks
Asha,78
Ravi,91
Meena,85


[1] "character"

name,marks
<chr>,<dbl>
Asha,78
Ravi,91
Meena,85


[1] "numeric"

## The `painters` data set (package **MASS**)

Package **MASS** provides functions and data sets supporting Venables & Ripley, *Modern Applied Statistics with S* (4th ed., 2002).  
`painters` records scores of 54 classical painters. The **names of the painters are row names**, not a variable.

In [ ]:
library(MASS)
head(painters, 10)     # the slides show an excerpt; use `painters` to print all 54 rows

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Da Udine,10,8,16,3,A
Da Vinci,15,16,4,14,A
Del Piombo,8,13,16,7,A
Del Sarto,12,16,9,8,A
Fr. Penni,0,15,8,0,A
Guilio Romano,15,16,4,14,A
Michelangelo,8,17,4,8,A
Perino del Vaga,15,16,7,6,A
Perugino,4,12,10,4,A


In [ ]:
dim(painters)          # 54 rows, 5 columns

[1] 54  5

### Row names (not a variable)

In [ ]:
rownames(painters)

[1] "Da Udine"        "Da Vinci"        "Del Piombo"      "Del Sarto"      
 [5] "Fr. Penni"       "Guilio Romano"   "Michelangelo"    "Perino del Vaga"
 [9] "Perugino"        "Raphael"         "F. Zucarro"      "Fr. Salviata"   
[13] "Parmigiano"      "Primaticcio"     "T. Zucarro"      "Volterra"       
[17] "Barocci"         "Cortona"         "Josepin"         "L. Jordaens"    
[21] "Testa"           "Vanius"          "Bassano"         "Bellini"        
[25] "Giorgione"       "Murillo"         "Palma Giovane"   "Palma Vecchio"  
[29] "Pordenone"       "Tintoretto"      "Titian"          "Veronese"       
[33] "Albani"          "Caravaggio"      "Corregio"        "Domenichino"    
[37] "Guercino"        "Lanfranco"       "The Carraci"     "Durer"          
[41] "Holbein"         "Pourbus"         "Van Leyden"      "Diepenbeck"     
[45] "J. Jordaens"     "Otho Venius"     "Rembrandt"       "Rubens"         
[49] "Teniers"         "Van Dyck"        "Bourdon"         "Le Brun"        
[53] "Le Suer"         "Poussin"

### Column names

In [ ]:
colnames(painters)

[1] "Composition" "Drawing"     "Colour"      "Expression"  "School"

### Variable types
Four **numeric** variables (`Composition`, `Drawing`, `Colour`, `Expression`) and one **factor** (`School`).  
Extract a column with the `$` operator.

In [ ]:
is.numeric(painters$School)
is.numeric(painters$Drawing)

[1] FALSE

[1] TRUE

In [ ]:
is.factor(painters$School)
is.factor(painters$Drawing)

[1] TRUE

[1] FALSE

### `summary()` of a data frame
Gives descriptive statistics per variable. For the factor `School`, only the **6 most frequent** levels are shown; the rest (F and H, 4 each) are lumped into `(Other): 8`.

In [ ]:
summary(painters)

  Composition       Drawing          Colour        Expression         School  
 Min.   : 0.00   Min.   : 6.00   Min.   : 0.00   Min.   : 0.000   A      :10  
 1st Qu.: 8.25   1st Qu.:10.00   1st Qu.: 7.25   1st Qu.: 4.000   D      :10  
 Median :12.50   Median :13.50   Median :10.00   Median : 6.000   E      : 7  
 Mean   :11.56   Mean   :12.46   Mean   :10.94   Mean   : 7.667   G      : 7  
 3rd Qu.:15.00   3rd Qu.:15.00   3rd Qu.:16.00   3rd Qu.:11.500   B      : 6  
 Max.   :18.00   Max.   :18.00   Max.   :18.00   Max.   :18.000   C      : 6  
                                                                  (Other): 8  

---
# Lecture 42 — Data Frames: Creation and Operations

This lecture re-introduces `painters` from MASS, where each **row is labelled with the painter's name**.  
(Also worth knowing — how to build a data frame yourself:)

In [ ]:
# Creating a data frame from scratch
students = data.frame(
  roll   = 1:4,
  name   = c("Asha", "Ravi", "Meena", "Kiran"),
  marks  = c(78, 91, 85, 66),
  passed = c(TRUE, TRUE, TRUE, FALSE)
)
students
str(students)      # structure: type of each column

roll,name,marks,passed
<int>,<chr>,<dbl>,<lgl>
1,Asha,78,TRUE
2,Ravi,91,TRUE
3,Meena,85,TRUE
4,Kiran,66,FALSE


'data.frame':	4 obs. of  4 variables:
 $ roll  : int  1 2 3 4
 $ name  : chr  "Asha" "Ravi" "Meena" "Kiran"
 $ marks : num  78 91 85 66
 $ passed: logi  TRUE TRUE TRUE FALSE


In [ ]:
library(MASS)
painters[1:6, ]           # first six painters

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Da Udine,10,8,16,3,A
Da Vinci,15,16,4,14,A
Del Piombo,8,13,16,7,A
Del Sarto,12,16,9,8,A
Fr. Penni,0,15,8,0,A
Guilio Romano,15,16,4,14,A


---
# Lecture 43 — Data Frames: More Operations

## 1. `summary()` of a categorical variable
Returns a **frequency table**.

In [ ]:
summary(painters$School)

A  B  C  D  E  F  G  H 
10  6  6 10  7  4  7  4

## 2. `attach()` and `detach()`

`attach(df)` lets you use the variable names **directly**, without the `painters$` prefix.

In [ ]:
attach(painters)

In [ ]:
summary(School)          # categorical variable
summary(Composition)     # numeric variable

A  B  C  D  E  F  G  H 
10  6  6 10  7  4  7  4

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    8.25   12.50   11.56   15.00   18.00 

`detach()` restores the default — afterwards you must use `painters$` again.

In [ ]:
detach(painters)

tryCatch(
  summary(School),
  error = function(e) cat("Error:", conditionMessage(e), "\n")
)

Error: object 'School' not found 


## 3. `subset()` — rows that satisfy a condition

`==` is the logical *equal-to* operator.

In [ ]:
subset(painters, School == "F")

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Durer,8,10,10,8,F
Holbein,9,10,16,13,F
Pourbus,4,15,6,6,F
Van Leyden,8,6,6,4,F


An equivalent way, using logical indexing (note the trailing comma = *all columns*):

In [ ]:
painters[painters[["School"]] == "F", ]

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Durer,8,10,10,8,F
Holbein,9,10,16,13,F
Pourbus,4,15,6,6,F
Van Leyden,8,6,6,4,F


Another condition:

In [ ]:
subset(painters, Composition <= 6)

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Fr. Penni,0,15,8,0,A
Perugino,4,12,10,4,A
Bassano,6,8,17,0,D
Bellini,4,6,14,0,D
Murillo,6,8,15,4,D
Palma Vecchio,5,6,16,0,D
Caravaggio,6,6,16,0,E
Pourbus,4,15,6,6,F


### Dropping uninteresting columns with `select`
Negative indices remove columns — here the 3rd (`Colour`) and 5th (`School`).

In [ ]:
subset(painters, School == "F", select = c(-3, -5))

,Composition,Drawing,Expression
,<int>,<int>,<int>
Durer,8,10,8
Holbein,9,10,13
Pourbus,4,15,6
Van Leyden,8,6,4


## 4. `split()` — partition by a (factor) variable

Splits the data set by the values of a variable; preferably a **factor**. The result is a **list of data frames**, one per level.

In [ ]:
splitted = split(painters, painters$School)
splitted

,Composition,Drawing,Colour,Expression,School
,<int>,<int>,<int>,<int>,<fct>
Da Udine,10,8,16,3,A
Da Vinci,15,16,4,14,A
Del Piombo,8,13,16,7,A
Del Sarto,12,16,9,8,A
Fr. Penni,0,15,8,0,A
Guilio Romano,15,16,4,14,A
Michelangelo,8,17,4,8,A
Perino del Vaga,15,16,7,6,A
Perugino,4,12,10,4,A


If the data are not attached, use `painters$School` as above.  Each element is itself a data frame:

In [ ]:
is.data.frame(splitted$A)
names(splitted)            # the list has one entry per School (A–H)

[1] TRUE

[1] "A" "B" "C" "D" "E" "F" "G" "H"

---
# Lecture 44 — Data Frames: Combining and Merging

| Technique | What it does |
|---|---|
| `cbind()` | combine columns of two data frames **side by side** |
| `merge()` | join two data frames using a **common column** |
| `rbind()` | **stack** one data frame under another |

## 1. `cbind()` — horizontal combination

Create two data frames:

In [ ]:
df1 = data.frame(state = c("UP", "MP", "AP", "JK"),
                 popnsize = c(1000, 2000, 3000, 4000))

df2 = data.frame(state = c("UP", "MP", "AP", "JK"),
                 samplesize = c(100, 200, 300, 400),
                 surveycompleted = c("Yes", "No", "Yes", "No"))
df1
df2

state,popnsize
<chr>,<dbl>
UP,1000
MP,2000
AP,3000
JK,4000


state,samplesize,surveycompleted
<chr>,<dbl>,<chr>
UP,100,Yes
MP,200,No
AP,300,Yes
JK,400,No


In [ ]:
cbind(df1, df2)     # note the duplicated `state` column

state,popnsize,state,samplesize,surveycompleted
<chr>,<dbl>,<chr>,<dbl>,<chr>
UP,1000,UP,100,Yes
MP,2000,MP,200,No
AP,3000,AP,300,Yes
JK,4000,JK,400,No


## 2. `merge()` — join on a common column

```r
merge(x, y, by, by.x, by.y, sort, no.dups, ...)
```

| Argument | Meaning |
|---|---|
| `x`, `y` | data frames (or objects coercible to one) |
| `by`, `by.x`, `by.y` | column(s) used for matching |
| `sort` | logical: sort the result on the `by` columns |
| `no.dups` | logical: append suffixes to avoid duplicate column names |

`state` is common to both, so we merge on it. The common column appears **once**, and the result is **sorted** by it.

In [ ]:
merge(df1, df2, by = "state")

state,popnsize,samplesize,surveycompleted
<chr>,<dbl>,<dbl>,<chr>
AP,3000,300,Yes
JK,4000,400,No
MP,2000,200,No
UP,1000,100,Yes


`by` can be omitted — `merge` then uses all columns with common names. Compare with `cbind` above: no duplicated `state` column.

In [ ]:
merge(df1, df2)                    # same result, `by` inferred
merge(df1, df2, by = "state", sort = FALSE)   # keep original order

state,popnsize,samplesize,surveycompleted
<chr>,<dbl>,<dbl>,<chr>
AP,3000,300,Yes
JK,4000,400,No
MP,2000,200,No
UP,1000,100,Yes


state,popnsize,samplesize,surveycompleted
<chr>,<dbl>,<dbl>,<chr>
UP,1000,100,Yes
MP,2000,200,No
AP,3000,300,Yes
JK,4000,400,No


## 3. `rbind()` — vertical combination

`rbind` stacks data frames; the **column names must match**.

In [ ]:
df11 = data.frame(state = c("UP", "MP", "AP", "JK"),
                  popnsize = c(1000, 2000, 3000, 4000))

df22 = data.frame(state = c("Bihar", "Delhi", "Punjab"),
                  popnsize = c(100, 200, 300))
df11
df22

state,popnsize
<chr>,<dbl>
UP,1000
MP,2000
AP,3000
JK,4000


state,popnsize
<chr>,<dbl>
Bihar,100
Delhi,200
Punjab,300


In [ ]:
rbind(df11, df22)

state,popnsize
<chr>,<dbl>
UP,1000
MP,2000
AP,3000
JK,4000
Bihar,100
Delhi,200
Punjab,300


---
# Quick Reference

### Strings (Lecture 40)
| Task | Code | Result |
|---|---|---|
| Replace first match | `sub("a","X","banana")` | `"bXnana"` |
| Replace all matches | `gsub("a","X","banana")` | `"bXnXnX"` |
| Indices of matches | `grep("ex", str)` | `2 3` |
| Matching elements | `grep("ex", str, value=TRUE)` | the strings |
| Ignore case | `grep("r", str, ignore.case=TRUE)` | |
| TRUE/FALSE per element | `grepl("ex", str)` | `FALSE TRUE TRUE` |

### Data frames (Lectures 41–44)
| Task | Code |
|---|---|
| Create | `data.frame(a = 1:3, b = c("x","y","z"))` |
| Column by name | `df$col` |
| Names | `rownames(df)`, `colnames(df)` |
| Type checks | `is.numeric(df$col)`, `is.factor(df$col)` |
| Overview | `summary(df)`; `summary(df$factor)` gives a frequency table |
| Use names directly | `attach(df)` … `detach(df)` |
| Filter rows | `subset(df, cond)` or `df[cond, ]` |
| Drop columns | `subset(df, cond, select = c(-3, -5))` |
| Partition by factor | `split(df, df$factor)` |
| Side by side | `cbind(df1, df2)` |
| Join on key | `merge(df1, df2, by = "key")` |
| Stack | `rbind(df1, df2)` |

In [ ]:
# Bonus check of the quick-reference examples
sub("a", "X", "banana")
gsub("a", "X", "banana")

[1] "bXnana"

[1] "bXnXnX"